<a href="https://colab.research.google.com/github/juanfebotero/juanfebotero.github.io/blob/main/notebooks/tecnicas-optimizacion/02_recocido_simulado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Recocido simulado para el problema del viajero (TSP)

**Técnicas de Optimización** · Prof. Juan Felipe Botero · Universidad de Antioquia

**Objetivos**

- Entender por qué el TSP es NP-difícil y cuándo usar metaheurísticas.
- Implementar recocido simulado con vecindario 2-opt.
- Estudiar el efecto de la temperatura inicial y el factor de enfriamiento.
- Comparar con la solución exacta en una instancia pequeña.

> **Antes de empezar:** use *Archivo → Guardar una copia en Drive*. Si el cuaderno le llegó por Google Classroom, ya es su copia personal y el profesor verá su progreso.

In [ ]:
import math, random, itertools
import numpy as np, matplotlib.pyplot as plt
random.seed(3)

## 1. Instancia
Ocho nodos de una red de fibra que un vehículo de mantenimiento debe visitar una vez y regresar al origen.

In [ ]:
ciudades = [(23.8, 54.4), (37.0, 60.4), (62.6, 6.6), (1.3, 83.7), (25.9, 23.4), (99.6, 47.0), (83.6, 47.6), (63.9, 15.1)]
def longitud(t):
    return sum(math.dist(ciudades[t[i]], ciudades[t[(i+1) % len(t)]]) for i in range(len(t)))
def dibujar(t, titulo=""):
    p = [ciudades[i] for i in t] + [ciudades[t[0]]]
    plt.figure(figsize=(4.5, 4.5)); plt.plot(*zip(*p), "-o"); plt.title(f"{titulo}  L = {longitud(t):.1f}"); plt.show()
dibujar(list(range(len(ciudades))), "Recorrido inicial")

## 2. ¿Por qué no probar todo?
Con $n$ ciudades hay $(n-1)!/2$ recorridos distintos. Para 8 son 2520 (fácil). Para 20 son $6 \times 10^{16}$.

In [ ]:
for n in [8, 12, 16, 20, 30]:
    print(n, f"{math.factorial(n-1)//2:.3e}")

## 3. Recocido simulado
Idea: aceptar a veces soluciones peores, con probabilidad $e^{-\Delta/T}$, para escapar de óptimos locales. La temperatura $T$ baja poco a poco: $T \leftarrow \alpha T$.

Vecindario **2-opt**: se invierte un tramo del recorrido.

In [ ]:
def vecino_2opt(t):
    i, j = sorted(random.sample(range(len(t)), 2))
    return t[:i] + t[i:j+1][::-1] + t[j+1:]

def recocido(t0, T0=100.0, alfa=0.995, iters=5000):
    actual, mejor = t0[:], t0[:]
    T, historia = T0, []
    for _ in range(iters):
        cand = vecino_2opt(actual)
        delta = longitud(cand) - longitud(actual)
        if delta < 0 or random.random() < math.exp(-delta / T):
            actual = cand
            if longitud(actual) < longitud(mejor):
                mejor = actual[:]
        historia.append(longitud(actual)); T *= alfa
    return mejor, historia

mejor, hist = recocido(list(range(len(ciudades))))
plt.plot(hist); plt.xlabel("iteración"); plt.ylabel("longitud"); plt.show()
dibujar(mejor, "Recocido simulado")

## 4. Ejercicios
**Ejercicio 1.** Calcule la longitud del recorrido **óptimo** por fuerza bruta (fije la ciudad 0 como inicio y permute las demás). Guárdela en `resp_1`.

In [ ]:
# Escriba aquí su solución
resp_1 = None

**Ejercicio 2.** Ejecute `recocido` 20 veces con `T0=1` y 20 veces con `T0=100`. ¿Cuál encuentra el óptimo más veces? Explique con sus palabras el papel de la temperatura inicial.

In [ ]:
# Experimento del ejercicio 2

**Ejercicio 3 (reto).** Genere 30 ciudades aleatorias y compare recocido simulado con una búsqueda local pura (solo acepta mejoras). Grafique ambas curvas de convergencia.

In [ ]:
# --- Autoevaluación: no modifique esta celda ---
import math
def verificar(nombre, valor, esperado, tol=1e-6):
    """Compara su respuesta con la esperada y muestra el resultado."""
    if valor is None:
        print(f"[ ] {nombre}: pendiente, complete la celda anterior.")
        return False
    try:
        ok = abs(float(valor) - float(esperado)) <= tol * max(1, abs(float(esperado)))
    except (TypeError, ValueError):
        ok = valor == esperado
    print(f"[{'✔' if ok else '✘'}] {nombre}: {'correcto' if ok else 'revise su respuesta (obtuvo ' + repr(valor) + ')'}")
    return ok

verificar('Ejercicio 1', resp_1, 271.8185986805791, tol=1e-3)